# Residual Network

The goal is use the model subclassing API together with custom layers to create a residual network architecture. Therefore, It's neccesary train a custom model on the Fashion-MNIST dataset by using a custom training loop and implementing the automatic differentiation tools in Tensorflow to calculate the gradients for backpropagation.

### Starting running some imports and loading the dataset

In [2]:
# Imports

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Layer, BatchNormalization, Conv2D, Dense,
                                     Flatten, Add)
import numpy as np
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

# Other imporst




![imagen](data/fashion_mnist.png)

## The Fashion MNIST dataset

The idea is use the [Fashion-MNIST dataset](https://github.com/zalandoresearch/fashion-mnist). It consists of a training set of 60,000 images of fashion items with corresponding labels, and a test set of 10,000 images. The images have been normalised and centred. The dataset is frequently used in machine learning research, especially as a drop-in replacement for the MNIST dataset. 

- H. Xiao, K. Rasul, and R. Vollgraf. "Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms." arXiv:1708.07747, August 2017.

The goal is to construct a ResNet model that classifies images of fashion items into one of 10 classes.


### Loading the dataset

In [4]:


(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

train_images =  train_images.astype(np.float32)
test_images = test_images.astype(np.float32)

train_images = train_images[:5000] / 255.
train_labels = train_labels[:5000]

test_images = test_images / 255.

train_images = train_images[..., np.newaxis]
test_images = test_images[..., np.newaxis]

print(test_images.shape)


(10000, 28, 28, 1)


In [5]:
# Creating the Dataset object for the training and test sets

train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_dataset = train_dataset.batch(32)


test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
test_dataset = test_dataset.batch(32)


In [6]:
#Getting the dataset labels

image_labels = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

### Creating custom layers for the residual blocks

In [7]:

class ResidualBlock(Layer):

    def __init__(self, **kwargs):
        super(ResidualBlock, self).__init__(**kwargs)


    def build(self, input_shape):
        filters = input_shape[-1]
        self.batch_layer1 = BatchNormalization(input_shape = input_shape)
        self.conv2d1 = Conv2D(filters=filters, kernel_size=(3,3), padding='SAME')
        self.batch_layer2 = BatchNormalization()
        self.conv2d2 = Conv2D(filters=filters, kernel_size=(3,3), padding='SAME')

    def call(self, inputs, training=False):    

        x = self.batch_layer1(inputs, training=training)
        x = tf.nn.relu(x)
        x = self.conv2d1(x)

        x = self.batch_layer2(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2d2(x)

        return inputs + x


In [8]:

test_model = tf.keras.Sequential([ResidualBlock(input_shape = (28,28,1),
                                                name= "residual_block")])
test_model.summary()

C:\Users\ramir\AppData\Local\Temp\ipykernel_28224\2580892015.py:4: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(ResidualBlock, self).__init__(**kwargs)
c:\Users\ramir\OneDrive\Documents\ESTUDIO\DEEP LEARNING\Tensorflow 2 for Deep Learning\PROYECTOS 2\tf2-Classifier-fashion-mnist-custom-training-loop\.venv\Lib\site-packages\keras\src\layers\normalization\batch_normalization.py:181: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ residual_block (ResidualBlock)  │ (None, 28, 28, 1)      │            28 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28 (112.00 B)

 Trainable params: 24 (96.00 B)

 Non-trainable params: 4 (16.00 B)

In [9]:


class FiltersChangeResidualBlock(Layer):

    def __init__(self, out_filters, **kwargs):
        super(FiltersChangeResidualBlock, self).__init__(**kwargs)
        self.out_filters = out_filters



    def build(self, input_shape):

        filters = input_shape[-1]
        self.batch_norma1 = BatchNormalization(input_shape=input_shape)
        self.conv2d1 = Conv2D(filters=filters, kernel_size=(3,3), padding="SAME")
        self.batch_norma2 = BatchNormalization()
        self.conv2d2 = Conv2D(filters=self.out_filters, kernel_size=(3,3), padding="SAME")
        self.conv2d3 = Conv2D(filters=self.out_filters, kernel_size=(1,1))

    
    def call(self, inputs, training = False):
        x = self.batch_norma1(inputs, training=training)
        x = tf.nn.relu(x)
        x = self.conv2d1(x)
        x = self.batch_norma2(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2d2(x)

        return x + self.conv2d3(inputs)





In [10]:

test_model = tf.keras.Sequential([FiltersChangeResidualBlock(16, input_shape=(32, 32, 3), name="fc_resnet_block")])
test_model.summary()

C:\Users\ramir\AppData\Local\Temp\ipykernel_28224\1990838982.py:4: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(FiltersChangeResidualBlock, self).__init__(**kwargs)
c:\Users\ramir\OneDrive\Documents\ESTUDIO\DEEP LEARNING\Tensorflow 2 for Deep Learning\PROYECTOS 2\tf2-Classifier-fashion-mnist-custom-training-loop\.venv\Lib\site-packages\keras\src\layers\normalization\batch_normalization.py:181: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc_resnet_block                 │ (None, 32, 32, 16)     │           620 │
│ (FiltersChangeResidualBlock)    │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 620 (2.42 KB)

 Trainable params: 608 (2.38 KB)

 Non-trainable params: 12 (48.00 B)